# Region extract → TOPCAT (EDFN bbox × DESI_DR1)

**Environment:** [uv](https://docs.astral.sh/uv/) project venv from the [Quick-start](../docs/quickstart.md) (`uv sync --extra dev`). Select kernel **Python (data-lake)**.

This notebook shows a short operator loop:

1. **Save an area** from a bounding box (Euclid Deep Field North / EDFN example)
2. **Extract** selected DESI_DR1 columns with `dl-extract-catalog --from-area`
3. **Send** the subset to a running TOPCAT with `dl-samp-send-catalog`

Companion docs: [export-and-sharing.md](../docs/export-and-sharing.md#send-to-topcat-via-samp), [regions-and-areas.md](../docs/discovery/regions-and-areas.md).

> Start **TOPCAT** before the SAMP cell so a hub is available. Set `SEND_TO_TOPCAT = False` to skip that step.

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

# Set DATA_LAKE_CONFIG before running, e.g.:
#   export DATA_LAKE_CONFIG=/path/to/lake_config.toml
os.environ.setdefault("DATA_LAKE_CONFIG", os.environ.get("DATA_LAKE_CONFIG", ""))

from data_lake.config import LakeConfig

cfg = LakeConfig.discover()
LAKE_ROOT = Path(cfg.lake.root)

# ── Edit for your run ─────────────────────────────────────────────────────────
AREA_ID = "EDFN"
SURVEY = "DESI_DR1"
# --bbox RA_MIN RA_MAX DEC_MIN DEC_MAX  (degrees, ICRS)
BBOX = (258.5, 281.03, 61.5, 70.3)
COLUMNS = "TARGETID,TARGET_RA,TARGET_DEC,Z"
SCRATCH = Path("/scratch/data_lake_edfn")  # writable path
SCRATCH.mkdir(parents=True, exist_ok=True)
EXTRACT_OUT = SCRATCH / f"{SURVEY}_{AREA_ID}.parquet"
SEND_TO_TOPCAT = True  # set False if TOPCAT is not running
OVERWRITE_AREA = True

print("Lake root:", LAKE_ROOT)
print("Area:", AREA_ID, "bbox:", BBOX)
print("Extract →", EXTRACT_OUT)

In [ ]:
def run_dl(*args: str, check: bool = True) -> subprocess.CompletedProcess:
    """Run a dl-* CLI command and print stdout/stderr."""
    cmd = ["dl-" + args[0], *[str(a) for a in args[1:]]]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"{' '.join(cmd)} failed (exit {proc.returncode})"
        )
    return proc

## 1. Create the EDFN area

`dl-region --bbox … --save-as` writes `areas/EDFN.json` under the lake. Re-run with `--overwrite` to replace it.

In [ ]:
ra_min, ra_max, dec_min, dec_max = BBOX
args = [
    "region", str(LAKE_ROOT),
    "--bbox", str(ra_min), str(ra_max), str(dec_min), str(dec_max),
    "--save-as", AREA_ID,
]
if OVERWRITE_AREA:
    args.append("--overwrite")
run_dl(*args)

from data_lake.discovery.areas import load_area

area = load_area(LAKE_ROOT, AREA_ID)
print("Saved:", LAKE_ROOT / "areas" / f"{AREA_ID}.json")
print("Region:", area.region.to_dict())

## 2. Discover overlap (optional)

Confirm `DESI_DR1` catalog tiles fall in the area before extracting.

In [ ]:
run_dl(
    "region", str(LAKE_ROOT),
    "--from-area", AREA_ID,
    "--surveys", SURVEY,
    "--modalities", "catalog",
)

## 3. Extract catalog columns in the area

Lake extract with `--from-area` prunes to overlapping HEALPix tiles, then writes Parquet.

In [ ]:
run_dl(
    "extract-catalog",
    "--lake-root", str(LAKE_ROOT),
    "--survey", SURVEY,
    "--from-area", AREA_ID,
    "-c", COLUMNS,
    "-o", str(EXTRACT_OUT),
)

import pyarrow.parquet as pq

tbl = pq.read_table(EXTRACT_OUT)
print(f"Rows: {tbl.num_rows:,}  Columns: {tbl.column_names}")
tbl.slice(0, min(5, tbl.num_rows)).to_pandas()

## 4. Send to TOPCAT via SAMP

Equivalent one-shot (extract + send without keeping the Parquet):

```bash
dl-samp-send-catalog "$LAKE" --survey DESI_DR1 --from-area EDFN \
  -c TARGETID,TARGET_RA,TARGET_DEC,Z --name DESI_EDFN
```

Here we reuse the file from §3 with `--input`.

In [ ]:
if not SEND_TO_TOPCAT:
    print("SEND_TO_TOPCAT=False — skipping SAMP. Open TOPCAT, set the flag True, re-run.")
else:
    run_dl(
        "samp-send-catalog",
        "--input", str(EXTRACT_OUT),
        "--name", f"{SURVEY}_{AREA_ID}",
    )
    print("Check TOPCAT for the new table.")

## 5. Optional: extract + send in one command

Skip the intermediate Parquet if you only need the TOPCAT view.

In [ ]:
# Uncomment to run the combined path (still requires TOPCAT):
# if SEND_TO_TOPCAT:
#     run_dl(
#         "samp-send-catalog", str(LAKE_ROOT),
#         "--survey", SURVEY,
#         "--from-area", AREA_ID,
#         "-c", COLUMNS,
#         "--name", f"{SURVEY}_{AREA_ID}_direct",
#         "--limit", "5000",  # optional cap for a quick look
#     )